<a href="https://colab.research.google.com/github/Ivan8Garcia/Proyectos_MachineLearning/blob/main/identificando_objetos_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import gymnasium as gym

# Inicializar el entorno FrozenLake
# El agente necesita cruzar un lago congelado sin caer en los agujeros
env = gym.make("FrozenLake-v1", is_slippery=True)

# Definir los hiperparámetros del Q-Learning
alpha = 0.8  # Tasa de aprendizaje: cuánto aprende el agente de nueva información
gamma = 0.95  # Factor de descuento: cuán importantes son las recompensas futuras en comparación con las inmediatas
epsilon = 1.0  # Probabilidad inicial de explorar acciones aleatorias
epsilon_decay = 0.999  # Reduce gradualmente la exploración a medida que el agente aprende
epsilon_min = 0.01  # Límite mínimo de exploración para garantizar que el agente aún explore un poco
num_episodes = 20000  # Número total de intentos de aprendizaje (episodios)

# Crear la tabla Q (Q-table) con ceros
# Las filas representan los estados y las columnas representan las acciones
q_table = np.zeros((env.observation_space.n, env.action_space.n))

# Iniciar el entrenamiento del agente
for episode in range(num_episodes):
    # Reiniciar el entorno en cada episodio
    state, _ = env.reset()
    done = False

    while not done:
        # Elegir una acción usando la estrategia epsilon-greedy
        # Con probabilidad 'epsilon', elegimos una acción aleatoria (exploración)
        # De lo contrario, elegimos la mejor acción conocida hasta el momento (explotación)
        if np.random.rand() < epsilon:
            action = env.action_space.sample()  # Explorar una acción aleatoria
        else:
            action = np.argmax(q_table[state])  # Explorar la acción con mayor valor en la Q-table

        # Ejecutar la acción elegida en el entorno
        next_state, reward, done, truncated, _ = env.step(action)

        # Actualizar la Q-table con la fórmula de aprendizaje por refuerzo
        # Q(s, a) = Q(s, a) + alpha * (recompensa + descuento * max(Q(s', a')) - Q(s, a))
        best_next_action = np.max(q_table[next_state])  # Mejor acción en el siguiente estado
        q_table[state, action] += alpha * (reward + gamma * best_next_action - q_table[state, action])

        # Avanzar al siguiente estado
        state = next_state

    # Reducir la tasa de exploración gradualmente, sin superar el mínimo definido
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

# Evaluar el rendimiento del agente entrenado
# Aquí, probamos 1000 episodios para verificar cuántas veces logra cruzar el lago con éxito
successes = 0
for episode in range(1000):
    state, _ = env.reset()
    done = False
    while not done:
        # El agente ahora solo elige la mejor acción aprendida (sin exploración aleatoria)
        action = np.argmax(q_table[state])
        state, reward, done, truncated, _ = env.step(action)
        if done and reward == 1.0:
            successes += 1  # Contabilizar los éxitos

# Mostrar el resultado final
print(f"El agente logró cruzar el lago con éxito en {successes} de 1000 episodios.")

El agente logró cruzar el lago con éxito en 756 de 1000 episodios.


Para mejorar el rendimiento del agente, puedes ajustar los hiperparámetros para incentivar más exploración al principio y, gradualmente, enfocarte en la explotación de las mejores acciones.

Por ejemplo, puedes implementar el siguiente cambio:

Disminución gradual de epsilon (exploración): Comienza con un valor más alto y disminúyelo con el tiempo.
Aumento de la tasa de aprendizaje (alpha): El agente aprende más rápido.
Más episodios de entrenamiento: Mayor oportunidad de aprender un comportamiento ideal.